In [39]:
import numpy as np
import pandas as pd
import fastf1
from pymongo import MongoClient
from time import sleep
fastf1.Cache.enable_cache(".fastf1_cache")
fastf1.set_log_level("ERROR")

import json
import random
from datetime import datetime, timedelta
from dataclasses import dataclass
from typing import Dict, List

In [40]:
year = 2025
track = "Monaco"
session = fastf1.get_session(year, track, "R")
session.load(telemetry=True)
drivers = session.drivers

In [ ]:
# ---- Helper data structures ----
@dataclass
class DriverMeta:
    id: str
    name: str
    code: str
    number: int
    team: str
    teamColorHex: str
    country: str

def generateDriversInfo(session) -> Dict[str, DriverMeta]:
    roster: Dict[str, DriverMeta] = {}
    driverIds = session.drivers

    for drv in driverIds:
        info = session.get_driver(drv)
        # FastF1 keys can vary a bit; use .get with sensible fallbacks
        code = info.get("Abbreviation") or info.get("Code") or drv
        number_raw = info.get("DriverNumber") or info.get("Number") or 0
        try:
            number = int(number_raw)
        except Exception:
            number = 0
        team = info.get("TeamName") or info.get("Team") or "Unknown"
        team_color = info.get("TeamColor") or info.get("TeamColorHex") or "000000"
        if not isinstance(team_color, str):
            team_color = str(team_color)
        team_color = team_color.upper().lstrip("#")
        country = info.get("CountryCode") or info.get("Country") or "N/A"
        name = info.get("FullName") or info.get("LastName") or info.get("BroadcastName") or drv
        roster[drv] = DriverMeta(
            id=f"{code}{number}",
            name=name,
            code=code,
            number=number,
            team=team,
            teamColorHex=f"#{team_color}",
            country=country,
        )
    return roster

In [43]:
driver = '4'

frequency = 10 # Hz
delta = pd.Timedelta(seconds=(1/frequency))
telemetry = session.laps.pick_drivers(driver).get_telemetry()
row_index = 1
start = pd.Timedelta(seconds=0) + 20000 * delta
while True:
# for _ in range(1):
    while telemetry.iloc[row_index]["Time"] < start:
        row_index += 1
    df_with_best_timestamp = telemetry.iloc[row_index - 1]

    # print(f"Current timestamp: {start}, \t\treal timestamp: {df_with_best_timestamp["Time"]}")
    # display(df_with_best_timestamp)
    start += delta

KeyboardInterrupt: 

In [47]:
freq = '100L'

laps = session.laps
race_start = laps['LapStartTime'].min()
race_end = (laps['LapStartTime'] + laps['LapTime']).max()
timeline = pd.timedelta_range(start=race_start, end=race_end, freq='100ms')  # was '100L'
# ---- Generate telemetry data ----
def driver_timeseries(driverId):
    tel = laps.pick_drivers(driverId).get_telemetry()
    tel = tel.set_index('SessionTime').sort_index()
    keep = [c for c in ['Speed','Throttle','Brake','nGear','DRS','RPM','LapNumber','X','Y','Z'] if c in tel.columns]
    tel = tel[keep]
    tel = tel.reindex(timeline, method='nearest', tolerance = pd.Timedelta("250ms"))
    tel['Driver'] = driverId
    return tel

drivers = sorted(laps['Driver'].dropna().unique())
series_list = [driver_timeseries(d) for d in drivers]
telemetry_ts = pd.concat(series_list).reset_index().rename(columns={'index': 'SessionTime'})




In [48]:
series_list

[                        Speed  Throttle Brake  nGear  DRS     RPM       X  \
 0 days 00:56:08.809000    0.0      21.0  True    1.0  1.0  8548.0 -7570.0   
 0 days 00:56:08.909000    0.0      21.0  True    1.0  1.0  8479.0 -7571.0   
 0 days 00:56:09.009000    0.0      21.0  True    1.0  1.0  8464.0 -7571.0   
 0 days 00:56:09.109000    0.0      21.0  True    1.0  1.0  8464.0 -7571.0   
 0 days 00:56:09.209000    0.0      21.0  True    1.0  1.0  8450.0 -7570.0   
 ...                       ...       ...   ...    ...  ...     ...     ...   
 0 days 02:37:54.409000    NaN       NaN   NaN    NaN  NaN     NaN     NaN   
 0 days 02:37:54.509000    NaN       NaN   NaN    NaN  NaN     NaN     NaN   
 0 days 02:37:54.609000    NaN       NaN   NaN    NaN  NaN     NaN     NaN   
 0 days 02:37:54.709000    NaN       NaN   NaN    NaN  NaN     NaN     NaN   
 0 days 02:37:54.809000    NaN       NaN   NaN    NaN  NaN     NaN     NaN   
 
                              Y      Z Driver  
 0 days 00:56: